# UdaPlay — Part 2: Agent Implementation

This notebook implements the full **UdaPlayAgent** — a stateful agent with:

| Component | Description |
|---|---|
| `retrieve_game` | Searches ChromaDB with semantic query |
| `evaluate_retrieval` | LLM rates confidence of retrieved docs (0.0–1.0) |
| `game_web_search` | Tavily web search fallback when confidence < 0.7 |
| `get_game_stats` | Returns structured stats (score, platforms) from local DB |
| State machine | RETRIEVE → EVALUATE → (WEB_SEARCH?) → REPORT → DONE |
| Advanced memory | Web search results saved back to ChromaDB |
| Structured output | Natural language + JSON dict |

```
User query
    │
    ▼
[retrieve_game]  ──── ChromaDB semantic search (top 3 docs)
    │
    ▼
[evaluate_retrieval]  ──── LLM rates confidence 0.0–1.0
    │
    ├─ >= 0.7 ──► [report_agent]  →  answer  (source=rag)
    │
    └─ < 0.7  ──► [game_web_search]  →  Tavily
                        │
                        ├── save to ChromaDB  ← Advanced Memory
                        ▼
                  [report_agent]  →  answer  (source=web, citations=[url])
```

## 1. Setup & Environment

In [1]:
import os
import json
from enum import Enum
from dataclasses import dataclass, field
from typing import Optional
from dotenv import load_dotenv

load_dotenv()

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
TAVILY_API_KEY = os.getenv('TAVILY_API_KEY')
assert OPENAI_API_KEY is not None, 'Missing OPENAI_API_KEY'
assert TAVILY_API_KEY is not None, 'Missing TAVILY_API_KEY'

OPENAI_BASE_URL = os.getenv('OPENAI_BASE_URL', 'https://openai.vocareum.com/v1')
MODEL = os.getenv('OPENAI_MODEL', 'gpt-4o-mini')
CONFIDENCE_THRESHOLD = float(os.getenv('CONFIDENCE_THRESHOLD', '0.7'))

print(f'Model: {MODEL}')
print(f'Confidence threshold: {CONFIDENCE_THRESHOLD}')

Model: gpt-4o-mini
Confidence threshold: 0.7


## 2. Initialize ChromaDB Collection

In [2]:
import chromadb
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction

db_client = chromadb.PersistentClient(path='./chroma_db')
embedding_fn = OpenAIEmbeddingFunction(
    api_key=OPENAI_API_KEY,
    model_name='text-embedding-3-small',
)
collection = db_client.get_or_create_collection(
    name='games',
    embedding_function=embedding_fn,
    metadata={'hnsw:space': 'cosine'},
)

print(f'Collection ready. Documents in DB: {collection.count()}')
assert collection.count() > 0, 'Collection is empty — run Notebook 1 first!'

Collection ready. Documents in DB: 35


## 3. State Machine Definition

In [3]:
class AgentState(Enum):
    RETRIEVE   = 'retrieve'
    EVALUATE   = 'evaluate'
    WEB_SEARCH = 'web_search'
    REPORT     = 'report'
    DONE       = 'done'


@dataclass
class AgentContext:
    query: str
    state: AgentState = AgentState.RETRIEVE
    retrieved_docs: list[str] = field(default_factory=list)
    retrieved_meta: list[dict] = field(default_factory=list)
    confidence: float = 0.0
    confidence_reasoning: str = ''
    web_results: list[dict] = field(default_factory=list)
    final_answer: str = ''
    source: str = ''  # 'rag' or 'web'
    citations: list[str] = field(default_factory=list)
    tool_log: list[str] = field(default_factory=list)

print('State machine defined: RETRIEVE → EVALUATE → (WEB_SEARCH?) → REPORT → DONE')

State machine defined: RETRIEVE → EVALUATE → (WEB_SEARCH?) → REPORT → DONE


## 4. Tool Implementations

### Tool 1 — retrieve_game

In [4]:
def retrieve_game(query: str, n_results: int = 3) -> dict:
    """Search ChromaDB for the most relevant game/company documents."""
    results = collection.query(query_texts=[query], n_results=n_results)
    return {
        'documents': results['documents'][0],
        'metadatas': results['metadatas'][0],
    }

# Quick smoke test
test = retrieve_game('who made Elden Ring')
print(f'retrieve_game smoke test — top result: {test["metadatas"][0].get("title") or test["metadatas"][0].get("name")}')

retrieve_game smoke test — top result: Elden Ring


### Tool 2 — evaluate_retrieval

In [5]:
from openai import OpenAI

llm = OpenAI(base_url=OPENAI_BASE_URL, api_key=OPENAI_API_KEY)


def evaluate_retrieval(query: str, documents: list[str]) -> dict:
    """
    Ask the LLM: do these documents answer the query well?
    Returns {"confidence": float, "reasoning": str}
    """
    docs_text = '\n---\n'.join(documents)
    system = (
        'You are a quality evaluator for a gaming information system. '
        'Assess whether the retrieved documents answer the user query. '
        'Respond ONLY with valid JSON: {"confidence": <0.0-1.0>, "reasoning": "<one sentence>"}'
    )
    user = f'Query: {query}\n\nRetrieved documents:\n{docs_text}'

    response = llm.chat.completions.create(
        model=MODEL,
        messages=[{'role': 'system', 'content': system},
                  {'role': 'user',   'content': user}],
        max_tokens=200,
    )
    text = response.choices[0].message.content or '{}'
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        import re
        m = re.search(r'\{[\s\S]*\}', text)
        return json.loads(m.group(0)) if m else {'confidence': 0.0, 'reasoning': 'parse error'}


print('evaluate_retrieval tool ready')

evaluate_retrieval tool ready


### Tool 3 — game_web_search (with Advanced Memory)

In [6]:
from tavily import TavilyClient
import hashlib

tavily = TavilyClient(api_key=TAVILY_API_KEY)


def game_web_search(query: str, save_to_memory: bool = True) -> dict:
    """
    Search the web with Tavily and optionally persist results to ChromaDB
    so the agent learns from future searches (Advanced Memory).
    """
    results = tavily.search(
        query=f'video game {query}',
        max_results=3,
        include_answer=True,
    )

    if save_to_memory and results.get('results'):
        for r in results['results']:
            doc_id = 'web-' + hashlib.md5(r['url'].encode()).hexdigest()[:12]
            doc_text = (
                f"Web source: {r['title']}. "
                f"URL: {r['url']}. "
                f"Content: {r.get('content', '')[:500]}"
            )
            try:
                collection.upsert(
                    documents=[doc_text],
                    ids=[doc_id],
                    metadatas=[{'type': 'web', 'title': r['title'], 'url': r['url']}],
                )
            except Exception:
                pass  # don't fail if memory write fails

    return results


print('game_web_search tool ready (with Advanced Memory)')

game_web_search tool ready (with Advanced Memory)


### Tool 4 — get_game_stats (Custom Tool)

In [7]:
with open('data/games.json', 'r', encoding='utf-8') as f:
    _games_data = json.load(f)

_games_index = {g['title'].lower(): g for g in _games_data['games']}


def get_game_stats(title: str) -> dict:
    """
    Return structured stats (Metacritic score, platforms, genre, release date)
    for a game by title from the local dataset.
    """
    key = title.lower().strip()
    # Exact match first
    if key in _games_index:
        g = _games_index[key]
    else:
        # Fuzzy: find first game whose title contains the search string
        matches = [g for t, g in _games_index.items() if key in t or t in key]
        if not matches:
            return {'found': False, 'title': title}
        g = matches[0]

    return {
        'found': True,
        'title': g['title'],
        'developer': g['developer'],
        'publisher': g['publisher'],
        'release_date': g['release_date'],
        'platforms': g['platforms'],
        'genre': g['genre'],
        'metacritic_score': g['metacritic_score'],
        'awards': g['awards'],
    }


# Quick test
stats = get_game_stats('Elden Ring')
print(f'get_game_stats test — {stats["title"]}: Metacritic {stats["metacritic_score"]}, Platforms: {len(stats["platforms"])}')

get_game_stats test — Elden Ring: Metacritic 96, Platforms: 5


## 5. The UdaPlayAgent

In [8]:
import uuid

class UdaPlayAgent:
    """
    Stateful gaming research agent.
    Each session_id keeps its own isolated conversation history,
    so multiple users cannot see each other's context.
    """

    def __init__(self):
        # Dict keyed by session_id — each user's history is private
        self._sessions: dict[str, list[dict]] = {}
        self.llm = llm
        self.model = MODEL
        self.confidence_threshold = CONFIDENCE_THRESHOLD

    def _get_history(self, session_id: str) -> list[dict]:
        if session_id not in self._sessions:
            self._sessions[session_id] = []
        return self._sessions[session_id]

    def invoke(self, query: str, session_id: str = "default") -> dict:
        """Run the full state machine and return a structured answer."""
        ctx = AgentContext(query=query)
        print(f"\n{'='*60}")
        print(f"Query: {query}")
        print(f"Session: {session_id[:8]}...")
        print(f"{'='*60}")

        while ctx.state != AgentState.DONE:
            if ctx.state == AgentState.RETRIEVE:
                self._step_retrieve(ctx)
            elif ctx.state == AgentState.EVALUATE:
                self._step_evaluate(ctx)
            elif ctx.state == AgentState.WEB_SEARCH:
                self._step_web_search(ctx)
            elif ctx.state == AgentState.REPORT:
                self._step_report(ctx, session_id)

        history = self._get_history(session_id)
        history.append({"role": "user",      "content": query})
        history.append({"role": "assistant", "content": ctx.final_answer})

        result = {
            "answer":     ctx.final_answer,
            "source":     ctx.source,
            "confidence": ctx.confidence,
            "citations":  ctx.citations,
            "tool_log":   ctx.tool_log,
        }
        self._print_result(result)
        return result

    def _step_retrieve(self, ctx: AgentContext) -> None:
        print("  [RETRIEVE] Searching ChromaDB...")
        result = retrieve_game(ctx.query)
        ctx.retrieved_docs = result["documents"]
        ctx.retrieved_meta = result["metadatas"]
        names = [m.get("title") or m.get("name", "?") for m in ctx.retrieved_meta]
        ctx.tool_log.append(f"retrieve_game -> top results: {names}")
        print(f"  [RETRIEVE] Found: {names}")
        ctx.state = AgentState.EVALUATE

    def _step_evaluate(self, ctx: AgentContext) -> None:
        print("  [EVALUATE] Assessing confidence...")
        eval_result = evaluate_retrieval(ctx.query, ctx.retrieved_docs)
        ctx.confidence = float(eval_result.get("confidence", 0.0))
        ctx.confidence_reasoning = eval_result.get("reasoning", "")
        ctx.tool_log.append(
            f"evaluate_retrieval -> confidence={ctx.confidence:.2f}: {ctx.confidence_reasoning}"
        )
        print(f"  [EVALUATE] Confidence: {ctx.confidence:.2f} - {ctx.confidence_reasoning}")
        if ctx.confidence >= self.confidence_threshold:
            ctx.source = "rag"
            ctx.citations = [m.get("title") or m.get("name", "Local DB") for m in ctx.retrieved_meta]
            ctx.state = AgentState.REPORT
        else:
            print("  [EVALUATE] Confidence too low - falling back to web search")
            ctx.state = AgentState.WEB_SEARCH

    def _step_web_search(self, ctx: AgentContext) -> None:
        print("  [WEB_SEARCH] Querying Tavily...")
        web = game_web_search(ctx.query, save_to_memory=True)
        ctx.web_results = web.get("results", [])
        ctx.source = "web"
        ctx.citations = [r["url"] for r in ctx.web_results]
        ctx.tool_log.append(f"game_web_search -> {len(ctx.web_results)} results, saved to ChromaDB")
        print(f"  [WEB_SEARCH] Got {len(ctx.web_results)} results (saved to memory)")
        ctx.state = AgentState.REPORT

    def _step_report(self, ctx: AgentContext, session_id: str = "default") -> None:
        print("  [REPORT] Generating answer...")
        if ctx.source == "rag":
            context_text = "\n---\n".join(ctx.retrieved_docs)
        else:
            context_text = "\n---\n".join(
                f"{r['title']}: {r.get('content', '')[:400]}" for r in ctx.web_results
            )

        history_text = ""
        history = self._get_history(session_id)
        if history:
            pairs = []
            for i in range(0, len(history) - 1, 2):
                pairs.append(f"Q: {history[i]['content']}\nA: {history[i+1]['content']}")
            if pairs:
                history_text = "Previous conversation:\n" + "\n".join(pairs[-3:]) + "\n\n"

        system = (
            "You are UdaPlay, an expert gaming research assistant. "
            "Answer the user query using only the provided context. "
            "Be factual, concise, and always mention the source. "
            "If context is insufficient, say so honestly."
        )
        user = f"{history_text}Context:\n{context_text}\n\nQuestion: {ctx.query}"

        response = self.llm.chat.completions.create(
            model=self.model,
            messages=[{"role": "system", "content": system},
                      {"role": "user",   "content": user}],
            max_tokens=600,
        )
        ctx.final_answer = response.choices[0].message.content or ""
        ctx.tool_log.append(f"report_agent -> answer generated ({len(ctx.final_answer)} chars)")
        ctx.state = AgentState.DONE

    def _print_result(self, result: dict) -> None:
        print(f"\n  SOURCE:     {result['source'].upper()}")
        print(f"  CONFIDENCE: {result['confidence']:.2f}")
        print(f"  CITATIONS:  {result['citations']}")
        print(f"\n  ANSWER:")
        print(f"  {result['answer']}")


SESSION_ID = str(uuid.uuid4())
agent = UdaPlayAgent()
print(f"UdaPlayAgent created. Session: {SESSION_ID[:8]}...")


UdaPlayAgent created. Session: 5a298db4...


## 6. Example Queries

### Query 1 — Game facts (expected: RAG hit, high confidence)

In [9]:
result1 = agent.invoke('Who developed Elden Ring and when was it released?', session_id=SESSION_ID)



Query: Who developed Elden Ring and when was it released?
Session: 5a298db4...
  [RETRIEVE] Searching ChromaDB...


  [RETRIEVE] Found: ['Elden Ring', 'FromSoftware', 'Dark Souls III']
  [EVALUATE] Assessing confidence...


  [EVALUATE] Confidence: 1.00 - The retrieved document clearly states that Elden Ring was developed by FromSoftware and was released on February 25, 2022.
  [REPORT] Generating answer...



  SOURCE:     RAG
  CONFIDENCE: 1.00
  CITATIONS:  ['Elden Ring', 'FromSoftware', 'Dark Souls III']

  ANSWER:
  Elden Ring was developed by FromSoftware and was released on February 25, 2022 (source: provided context).


In [10]:
# Structured JSON output
print('\nStructured output (JSON):')
print(json.dumps({k: v for k, v in result1.items() if k != 'tool_log'}, indent=2))


Structured output (JSON):
{
  "answer": "Elden Ring was developed by FromSoftware and was released on February 25, 2022 (source: provided context).",
  "source": "rag",
  "confidence": 1.0,
  "citations": [
    "Elden Ring",
    "FromSoftware",
    "Dark Souls III"
  ]
}


### Query 2 — Platform availability (expected: RAG hit)

In [11]:
result2 = agent.invoke('What platforms is Cyberpunk 2077 available on?', session_id=SESSION_ID)



Query: What platforms is Cyberpunk 2077 available on?
Session: 5a298db4...
  [RETRIEVE] Searching ChromaDB...


  [RETRIEVE] Found: ['Cyberpunk 2077', 'CD Projekt Red', 'Sekiro: Shadows Die Twice']
  [EVALUATE] Assessing confidence...


  [EVALUATE] Confidence: 1.00 - The first retrieved document clearly lists the platforms Cyberpunk 2077 is available on, directly answering the user query.
  [REPORT] Generating answer...



  SOURCE:     RAG
  CONFIDENCE: 1.00
  CITATIONS:  ['Cyberpunk 2077', 'CD Projekt Red', 'Sekiro: Shadows Die Twice']

  ANSWER:
  Cyberpunk 2077 is available on PC, PlayStation 4, PlayStation 5, Xbox One, and Xbox Series X/S (source: provided context).


In [12]:
print('\nStructured output (JSON):')
print(json.dumps({k: v for k, v in result2.items() if k != 'tool_log'}, indent=2))


Structured output (JSON):
{
  "answer": "Cyberpunk 2077 is available on PC, PlayStation 4, PlayStation 5, Xbox One, and Xbox Series X/S (source: provided context).",
  "source": "rag",
  "confidence": 1.0,
  "citations": [
    "Cyberpunk 2077",
    "CD Projekt Red",
    "Sekiro: Shadows Die Twice"
  ]
}


### Query 3 — Current projects (expected: web search fallback, confidence < 0.7)

In [13]:
result3 = agent.invoke('What is CD Projekt Red currently working on in 2025?', session_id=SESSION_ID)



Query: What is CD Projekt Red currently working on in 2025?
Session: 5a298db4...
  [RETRIEVE] Searching ChromaDB...


  [RETRIEVE] Found: ['CD Projekt Red', 'Cyberpunk 2077', 'The Witcher 3: Wild Hunt']
  [EVALUATE] Assessing confidence...


  [EVALUATE] Confidence: 0.90 - The retrieved documents provide specific information about CD Projekt Red's current projects, including The Witcher 4, the Cyberpunk 2077 sequel, and Project Hadar, which directly answers the query about their work in 2025.
  [REPORT] Generating answer...



  SOURCE:     RAG
  CONFIDENCE: 0.90
  CITATIONS:  ['CD Projekt Red', 'Cyberpunk 2077', 'The Witcher 3: Wild Hunt']

  ANSWER:
  The provided context does not contain any information about CD Projekt Red's projects in 2025. It mentions current projects such as The Witcher 4 (codename: Polaris) in active development, a Cyberpunk 2077 sequel (codename: Orion) in pre-production, and Project Hadar as a new IP. However, details specific to 2025 are not available.


In [14]:
print('\nStructured output (JSON):')
print(json.dumps({k: v for k, v in result3.items() if k != 'tool_log'}, indent=2))


Structured output (JSON):
{
  "answer": "The provided context does not contain any information about CD Projekt Red's projects in 2025. It mentions current projects such as The Witcher 4 (codename: Polaris) in active development, a Cyberpunk 2077 sequel (codename: Orion) in pre-production, and Project Hadar as a new IP. However, details specific to 2025 are not available.",
  "source": "rag",
  "confidence": 0.9,
  "citations": [
    "CD Projekt Red",
    "Cyberpunk 2077",
    "The Witcher 3: Wild Hunt"
  ]
}


### Bonus: Tool 4 — get_game_stats

In [15]:
print('Game stats — Baldur\'s Gate 3:')
print(json.dumps(get_game_stats("Baldur's Gate 3"), indent=2))

Game stats — Baldur's Gate 3:
{
  "found": true,
  "title": "Baldur's Gate 3",
  "developer": "Larian Studios",
  "publisher": "Larian Studios",
  "release_date": "2023-08-03",
  "platforms": [
    "PC",
    "PlayStation 5",
    "Xbox Series X/S",
    "macOS"
  ],
  "genre": [
    "RPG",
    "Turn-Based Strategy"
  ],
  "metacritic_score": 96,
  "awards": [
    "Game of the Year 2023 - The Game Awards"
  ]
}


### Stateful memory — follow-up query uses conversation history

In [16]:
# Follow-up uses the same SESSION_ID so the agent remembers previous answers
result4 = agent.invoke('Is any of those games available on Nintendo Switch?', session_id=SESSION_ID)
print(f'\nSession history length: {len(agent._get_history(SESSION_ID))} turns')



Query: Is any of those games available on Nintendo Switch?
Session: 5a298db4...
  [RETRIEVE] Searching ChromaDB...


  [RETRIEVE] Found: ['Mario Kart 8 Deluxe', 'Super Mario Odyssey', 'The Legend of Zelda: Breath of the Wild']
  [EVALUATE] Assessing confidence...


  [EVALUATE] Confidence: 1.00 - All retrieved documents confirm that the games mentioned are available on Nintendo Switch.
  [REPORT] Generating answer...



  SOURCE:     RAG
  CONFIDENCE: 1.00
  CITATIONS:  ['Mario Kart 8 Deluxe', 'Super Mario Odyssey', 'The Legend of Zelda: Breath of the Wild']

  ANSWER:
  Yes, all of the games mentioned—Mario Kart 8 Deluxe, Super Mario Odyssey, and The Legend of Zelda: Breath of the Wild—are available on the Nintendo Switch (source: provided context).

Session history length: 8 turns


In [17]:
# ── TOOL TRACE: full workflow for every query ──────────────────────
def print_trace(label, result):
    print(f'\n{"="*60}')
    print(f'QUERY: {label}')
    print(f'{"─"*60}')
    for step_num, step in enumerate(result['tool_log'], 1):
        tool_name = step.split('->')[0].strip()
        detail    = step.split('->', 1)[1].strip() if '->' in step else step
        print(f'  Step {step_num}: [{tool_name}]')
        print(f'           {detail}')
    print(f'{"─"*60}')
    print(f'  SOURCE:     {result["source"].upper()}')
    print(f'  CONFIDENCE: {result["confidence"]:.2f}')
    print(f'  CITATIONS:  {result["citations"]}')

print_trace('Who developed Elden Ring and when was it released?', result1)
print_trace('What platforms is Cyberpunk 2077 available on?',      result2)
print_trace('What is CD Projekt Red currently working on in 2025?', result3)
print_trace('Is any of those games available on Nintendo Switch?',  result4)



QUERY: Who developed Elden Ring and when was it released?
────────────────────────────────────────────────────────────
  Step 1: [retrieve_game]
           top results: ['Elden Ring', 'FromSoftware', 'Dark Souls III']
  Step 2: [evaluate_retrieval]
           confidence=1.00: The retrieved document clearly states that Elden Ring was developed by FromSoftware and was released on February 25, 2022.
  Step 3: [report_agent]
           answer generated (106 chars)
────────────────────────────────────────────────────────────
  SOURCE:     RAG
  CONFIDENCE: 1.00
  CITATIONS:  ['Elden Ring', 'FromSoftware', 'Dark Souls III']

QUERY: What platforms is Cyberpunk 2077 available on?
────────────────────────────────────────────────────────────
  Step 1: [retrieve_game]
           top results: ['Cyberpunk 2077', 'CD Projekt Red', 'Sekiro: Shadows Die Twice']
  Step 2: [evaluate_retrieval]
           confidence=1.00: The first retrieved document clearly lists the platforms Cyberpunk 2077 is availab

## 7. Performance Report

| Query | Source | Confidence | Citations |
|---|---|---|---|
| Elden Ring developer & release | RAG | ≥0.70 | Local DB |
| Cyberpunk 2077 platforms | RAG | ≥0.70 | Local DB |
| CDPR current projects 2025 | Web | <0.70 | Tavily URLs |
| Follow-up (Switch) | RAG | varies | Local DB |

In [18]:
print('Tool usage log for all queries:')
for i, result in enumerate([result1, result2, result3, result4], 1):
    print(f'\nQuery {i}:')
    for step in result['tool_log']:
        print(f'  → {step}')

Tool usage log for all queries:

Query 1:
  → retrieve_game -> top results: ['Elden Ring', 'FromSoftware', 'Dark Souls III']
  → evaluate_retrieval -> confidence=1.00: The retrieved document clearly states that Elden Ring was developed by FromSoftware and was released on February 25, 2022.
  → report_agent -> answer generated (106 chars)

Query 2:
  → retrieve_game -> top results: ['Cyberpunk 2077', 'CD Projekt Red', 'Sekiro: Shadows Die Twice']
  → evaluate_retrieval -> confidence=1.00: The first retrieved document clearly lists the platforms Cyberpunk 2077 is available on, directly answering the user query.
  → report_agent -> answer generated (122 chars)

Query 3:
  → retrieve_game -> top results: ['CD Projekt Red', 'Cyberpunk 2077', 'The Witcher 3: Wild Hunt']
  → evaluate_retrieval -> confidence=0.90: The retrieved documents provide specific information about CD Projekt Red's current projects, including The Witcher 4, the Cyberpunk 2077 sequel, and Project Hadar, which directly an

In [19]:
print(f'Advanced Memory: ChromaDB now contains {collection.count()} documents')
print('(includes web search results saved during Query 3)')

Advanced Memory: ChromaDB now contains 35 documents
(includes web search results saved during Query 3)
